# Model and training pipeline

Builds and checks the segmentation pipeline piece by piece: dataset, model,
loss, and the training loop. Each step is verified on a small slice before
moving on.

This notebook proves the pipeline runs. The full training run happens on a GPU
in Colab (see `notebooks/03_train_colab.ipynb`), because a real run is too slow
on a laptop.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader

SEG = Path("../data/Segmentation")

## 1. Dataset

Load one example and confirm the tensors are the right shape and range: image
scaled to 0-1 with 3 channels, mask as clean 0/1 integers.

In [2]:
from src.dataset import FishDataset

ds = FishDataset(SEG / "train.csv", SEG)
print("Dataset length:", len(ds))

img, msk = ds[0]
print(f"image: {img.shape}, {img.dtype}, range {img.min().item():.4f} to {img.max().item():.4f}")
print(f"mask : {msk.shape}, {msk.dtype}, values {torch.unique(msk).tolist()}")

Dataset length: 310
image: torch.Size([3, 256, 448]), torch.float32, range 0.0627 to 0.9882
mask : torch.Size([256, 448]), torch.int64, values [0, 1]


## 2. Model

DeepLabV3 with a ResNet-50 backbone, pretrained, with the final head swapped
to 2 classes (fish, background). The output channel should be 2.

In [3]:
from src.model import build_model

model = build_model(num_classes=2)
model.eval()

batch = img.unsqueeze(0)  # add a batch dimension
with torch.no_grad():
    out = model(batch)["out"]

print("input :", batch.shape)
print("output:", out.shape)
print("target:", msk.shape)

input : torch.Size([1, 3, 256, 448])
output: torch.Size([1, 2, 256, 448])
target: torch.Size([256, 448])


## 3. Data loaders

The loader batches and shuffles the examples for
training. Training is shuffled, validation is not.

In [4]:
train_ds = FishDataset(SEG / "train.csv", SEG)
val_ds = FishDataset(SEG / "val.csv", SEG)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

images, masks = next(iter(train_loader))
print("batch images:", images.shape)
print("batch masks :", masks.shape)

batch images: torch.Size([4, 3, 256, 448])
batch masks : torch.Size([4, 256, 448])


## 4. Loss and optimizer

Cross-entropy compares the model's per-pixel scores against the true mask.
Adam adjusts the weights.

In [5]:
model = build_model(num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.eval()
with torch.no_grad():
    out = model(images)["out"]
print("loss:", criterion(out, masks).item())

loss: 0.6696406006813049


## 5. Device

Picks the available accelerator.
The same code runs unchanged on either machine.

In [6]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


## 6. Smoke test one full epoch cycle

Confirm the epoch loop and the validation pass work together, and that the best
model is saved when validation improves. Still capped, so this is a check, not
the real run.

In [8]:
from src.train import train_one_epoch
from src.train import evaluate

model.to(device)
best_val = float("inf")
EPOCHS = 2

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device, max_batches=10
    )
    val_loss = evaluate(model, val_loader, criterion, device, max_batches=10)
    print(f"epoch {epoch + 1}/{EPOCHS}: train {train_loss:.4f}, val {val_loss:.4f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  saved best model, val {best_val:.4f}")

epoch 1/2: train 0.4751, val 0.5529
  saved best model, val 0.5529


epoch 2/2: train 0.3633, val 0.3799
  saved best model, val 0.3799
